## Database Notebook

This notebook created the sqlite database using clean csv files from the data folder. 

In [1]:
import pandas as pd
import sqlite3
import utils

In [2]:
connection = sqlite3.connect("../database/lfpl_oss_household_demographics.db")
cursor = connection.cursor()

In [3]:
connection.execute("DROP TABLE IF EXISTS louisville_zipcodes;")

connection.execute("""
CREATE TABLE louisville_zipcodes (
    zipcode TEXT PRIMARY KEY
);                   
""")
connection.commit()

In [4]:
connection.execute('DROP TABLE IF EXISTS libraries;')

connection.execute('''
CREATE TABLE libraries (
    library_id INT PRIMARY KEY,
    library_name TEXT, 
    latitude REAL,
    longitude REAL,
    zipcode TEXT,
    FOREIGN KEY (zipcode) REFERENCES louisville_zipcodes(zipcode)
);
''')
connection.commit()

In [5]:
connection.execute('DROP TABLE IF EXISTS library_item_details;')

connection.execute('''
CREATE TABLE library_item_details(
    item_id INT PRIMARY KEY,
    title TEXT,
    item_type TEXT,
    item_collection TEXT,
    item_location TEXT,
    item_price INT
);
''')
connection.commit()

In [6]:
connection.execute("DROP TABLE IF EXISTS library_inventory;")

connection.execute("""
CREATE TABLE library_inventory (
    library_id INT NOT NULL,
    item_id INT NOT NULL,
    PRIMARY KEY (library_id, item_id),
    FOREIGN KEY (library_id) REFERENCES libraries(library_id),
    FOREIGN KEY (item_id) REFERENCES library_item_details(item_id)
);
""")

connection.commit()

In [7]:
connection.execute('DROP TABLE IF EXISTS oss_households;')

connection.execute('''
CREATE TABLE oss_households (
    household_id INT PRIMARY KEY,
    date_added TEXT,
    household_type TEXT,
    household_size INT,
    annual_income BIGINT,
    zipcode TEXT,
    FOREIGN KEY (zipcode) REFERENCES louisville_zipcodes(zipcode)
);
''')

connection.commit()

In [8]:
louisville_zipcodes = utils.csv_to_sql('louisville_zipcodes', '../data/Clean/clean_zips.csv', connection, pd)

In [9]:
libraries = utils.csv_to_sql('libraries', '../data/Clean/clean_lfpl_loc.csv', connection, pd)


In [10]:
library_item_details = utils.csv_to_sql('library_item_details', '../data/Clean/clean_lfpl_inventory.csv', connection, pd)


In [11]:
libraries['library_name'] = libraries['library_name'].str.strip().str.upper()
library_item_details['item_location'] = library_item_details['item_location'].str.strip().str.upper()

In [12]:
library_inventory = library_item_details.merge(
    libraries[['library_id', 'library_name']],
    left_on='item_location',
    right_on='library_name',
    how='inner'
)

In [13]:
library_inventory = library_inventory[['library_id', 'item_id']]

In [14]:
library_inventory = library_inventory.drop_duplicates()

In [15]:
library_inventory.to_sql('library_inventory', connection, if_exists='replace', index=False)

1085827

In [16]:
oss_households = utils.csv_to_sql('oss_households', '../data/Clean/clean_oss.csv', connection, pd)

In [17]:
connection.execute("PRAGMA foreign_keys = ON;")